# Vector Embeddings: From Intuition to Semantic Search

**Chapter 01 of [ai-musings](../README.md)**

By the end of this notebook you will be able to:

1. Explain what an embedding is and why "close vectors mean similar meaning".
2. Implement **dot product**, **cosine similarity** and **Euclidean distance** from scratch.
3. See why **one-hot** vectors can't capture meaning, and learn **dense word vectors** from raw text.
4. Embed sentences with a **pretrained transformer model**.
5. Build a small **semantic search** engine.
6. **Visualize** high-dimensional embeddings in 2-D and cluster them.

> Run cells top to bottom (`Shift + Enter`). Every section ends with a short
> *Try it* prompt. Change something and re-run, because that's where the learning happens.

In [ ]:
# Setup: all of these are in requirements.txt
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["axes.grid"] = True

---
## Part 1: The intuition

An **embedding** is a list of numbers that describes something. Before we let a
neural network pick the numbers, let's pick them **by hand** so the idea is
concrete.

We'll describe each item with 3 hand-chosen features:

| feature | meaning |
|---|---|
| `is_animal` | 1 = definitely an animal, 0 = definitely not |
| `size` | 0 = tiny, 1 = huge |
| `is_food` | 1 = commonly eaten, 0 = not |

Each item becomes a point in 3-D space, and that point is its *embedding*.

In [ ]:
items = {
    #            is_animal  size  is_food
    "cat":       [0.95,     0.20, 0.05],
    "dog":       [0.95,     0.35, 0.05],
    "lion":      [0.95,     0.60, 0.00],
    "elephant":  [0.95,     0.95, 0.00],
    "chicken":   [0.90,     0.15, 0.80],
    "apple":     [0.00,     0.10, 0.95],
    "banana":    [0.00,     0.10, 0.90],
    "pizza":     [0.00,     0.25, 1.00],
    "car":       [0.00,     0.60, 0.00],
    "bus":       [0.00,     0.90, 0.00],
}
names = list(items)
X = np.array([items[n] for n in names])
print("Embedding matrix shape:", X.shape, "-> (num_items, num_dimensions)")
print("cat =", X[names.index("cat")])

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3-D projection)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(projection="3d")
ax.scatter(X[:, 0], X[:, 1], X[:, 2], s=60)
for name, (a, s, f) in zip(names, X):
    ax.text(a, s, f, f"  {name}", fontsize=9)
ax.set_xlabel("is_animal"); ax.set_ylabel("size"); ax.set_zlabel("is_food")
ax.set_title("Hand-made 3-D embeddings")
plt.show()

Notice the **clusters**: animals group together, fruit and food group together,
and vehicles group together. *Chicken* sits between animals and food, which is
fair.

That's the core promise of embeddings: **geometry encodes meaning.**

Real embedding models do the same thing with hundreds of dimensions. They
**learn** the features from data instead of having a human choose them, so no
single dimension has a tidy name like `size`.

> **Try it:** add `"shark"`, `"cookie"` and `"bicycle"` to `items` and re-run. Do they land where you expect?

---
## Part 2: Measuring similarity

"Close together" needs a number. There are three common measures:

| Measure | Formula | Range | Intuition |
|---|---|---|---|
| **Dot product** | $a \cdot b = \sum_i a_i b_i$ | $(-\infty, \infty)$ | Big when the vectors point the same way *and* are long |
| **Cosine similarity** | $\dfrac{a \cdot b}{\lVert a\rVert\,\lVert b\rVert}$ | $[-1, 1]$ | Only the **angle** counts; length is ignored |
| **Euclidean distance** | $\sqrt{\sum_i (a_i-b_i)^2}$ | $[0, \infty)$ | Straight-line distance. **Lower** means more similar |

Cosine similarity is the default for text embeddings, because a vector's
*direction* carries the meaning and its length mostly doesn't.

Let's implement all three with plain NumPy.

In [ ]:
def dot(a, b):
    return float(np.sum(a * b))

def norm(a):
    return float(np.sqrt(np.sum(a * a)))

def cosine_similarity(a, b):
    return dot(a, b) / (norm(a) * norm(b))

def euclidean_distance(a, b):
    return float(np.sqrt(np.sum((a - b) ** 2)))

v = lambda name: X[names.index(name)]

for pair in [("cat", "dog"), ("cat", "elephant"), ("cat", "apple"), ("apple", "banana"), ("car", "bus")]:
    a, b = v(pair[0]), v(pair[1])
    print(f"{pair[0]:>8} vs {pair[1]:<8}  dot={dot(a, b):5.2f}   cos={cosine_similarity(a, b):5.2f}   dist={euclidean_distance(a, b):5.2f}")

### Check our work against scikit-learn

Never trust a hand-rolled implementation until it matches a library.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as sk_cos

ours = np.array([[cosine_similarity(a, b) for b in X] for a in X])
theirs = sk_cos(X)
print("Max difference vs sklearn:", np.abs(ours - theirs).max())
assert np.allclose(ours, theirs)

### Why cosine ignores length

Scale a vector up and its **direction** doesn't change, so cosine similarity
stays the same while dot product and distance both shift. This matters because
a longer document shouldn't count as "more similar" just for being longer.

In [ ]:
cat = v("cat")
big_cat = cat * 10   # same direction, 10x longer

print("cos(cat, 10*cat)  =", round(cosine_similarity(cat, big_cat), 4))
print("dot(cat, 10*cat)  =", round(dot(cat, big_cat), 4), " vs dot(cat, cat) =", round(dot(cat, cat), 4))
print("dist(cat, 10*cat) =", round(euclidean_distance(cat, big_cat), 4))

**Pro tip:** if you **normalize** every vector to length 1 first, then
*dot product = cosine similarity*. Vector databases rely on this trick because a
dot product is cheaper to compute.

In [ ]:
unit = lambda a: a / norm(a)
a, b = v("apple"), v("pizza")
print("cosine(a, b)          =", round(cosine_similarity(a, b), 6))
print("dot(unit(a), unit(b)) =", round(dot(unit(a), unit(b)), 6))

In [ ]:
# The full similarity matrix as a heatmap
def plot_similarity(matrix, labels, title):
    fig, ax = plt.subplots(figsize=(0.55 * len(labels) + 2, 0.55 * len(labels) + 1.5))
    im = ax.imshow(matrix, cmap="viridis", vmin=min(0, matrix.min()), vmax=1)
    ax.set_xticks(range(len(labels)), labels, rotation=60, ha="right")
    ax.set_yticks(range(len(labels)), labels)
    ax.grid(False)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if matrix[i, j] < 0.6 else "black")
    fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

plot_similarity(ours, names, "Cosine similarity: hand-made embeddings")

> **Try it:** write `most_similar(name, k=3)`, which returns the `k` nearest
> items to `name` by cosine similarity. *(You'll reuse this idea all the way to
> semantic search.)*

---
## Part 3: From one-hot to *learned* vectors

Hand-picking features doesn't scale. How does a computer find useful numbers
on its own?

### The naive approach: one-hot vectors

Give each word in the vocabulary its own dimension: `cat = [1,0,0,...]`,
`dog = [0,1,0,...]`.

In [ ]:
vocab = ["cat", "dog", "kitten", "puppy", "car", "truck"]
one_hot = np.eye(len(vocab))
print(one_hot)
print("\ncos(cat, dog)    =", cosine_similarity(one_hot[0], one_hot[1]))
print("cos(cat, truck)  =", cosine_similarity(one_hot[0], one_hot[5]))

Every pair of different words scores **exactly 0**. One-hot vectors are all at
right angles to each other (*orthogonal*), so "cat" is as far from "dog" as it
is from "truck". They also need one dimension per word, which means 50,000+
dimensions for a real vocabulary.

We need **dense** vectors where related words share signal.

### The distributional hypothesis

> *"You shall know a word by the company it keeps."* (J. R. Firth, 1957)

Words that show up in similar **contexts** tend to have similar **meanings**.
"Cat" and "dog" both appear near *pet, fed, vet, cute*. "Car" and "truck"
appear near *drove, road, engine*.

We can turn that idea into vectors in three steps:

1. **Count** how often each word appears near every other word, which gives a *co-occurrence matrix*.
2. **Compress** that big sparse matrix with **SVD** (singular value decomposition) into a few dense dimensions.
3. Each word's row in the compressed matrix is its **learned embedding**.

This is essentially how early methods like LSA and GloVe worked. word2vec
reaches a similar result by training a small neural net.

In [ ]:
corpus = '''
the cat sat on the mat . the dog sat on the rug .
my cat is a cute pet . my dog is a loyal pet .
the kitten played with the cat . the puppy played with the dog .
i fed the cat and the kitten . i fed the dog and the puppy .
the vet checked the cat . the vet checked the puppy .
a kitten is a young cat . a puppy is a young dog .
i drove the car on the road . he drove the truck on the highway .
the car needs fuel . the truck needs fuel .
the car has an engine . the truck has a big engine .
we parked the car in the garage . they parked the truck near the garage .
the bus drove on the road . the bus has a big engine .
i ate an apple and a banana . she ate a banana and an orange .
the apple is a sweet fruit . the orange is a sweet fruit .
fresh fruit like apple and orange is healthy . banana is a healthy fruit .
'''.split()

stopwords = {"the", "a", "an", "is", "on", "and", "with", "i", "my", "he", "she",
             "we", "they", "in", "has", "needs", "like", "near", "."}

words = sorted({w for w in corpus if w not in stopwords})
idx = {w: i for i, w in enumerate(words)}
print(f"{len(corpus)} tokens, vocabulary of {len(words)} content words")

In [ ]:
# Step 1: count co-occurrences within a sliding window
WINDOW = 4
C = np.zeros((len(words), len(words)))
for i, w in enumerate(corpus):
    if w not in idx:
        continue
    for j in range(max(0, i - WINDOW), min(len(corpus), i + WINDOW + 1)):
        c = corpus[j]
        if j != i and c in idx:
            C[idx[w], idx[c]] += 1

print("Co-occurrence matrix:", C.shape)
print("Top contexts for 'cat':  ", [words[k] for k in np.argsort(-C[idx['cat']])[:6]])
print("Top contexts for 'truck':", [words[k] for k in np.argsort(-C[idx['truck']])[:6]])

In [ ]:
# Step 2: dampen raw counts (log), then compress with SVD
M = np.log1p(C)
U, S, Vt = np.linalg.svd(M)

DIMS = 5
word_vectors = U[:, :DIMS] * S[:DIMS]     # step 3: each row is a dense word embedding
print(f"Compressed {M.shape[1]}-D count vectors -> {DIMS}-D dense embeddings")
print("cat   =", word_vectors[idx["cat"]])
print("truck =", word_vectors[idx["truck"]])

In [ ]:
def nearest_words(word, k=4):
    q = word_vectors[idx[word]]
    sims = [(w, cosine_similarity(q, word_vectors[idx[w]])) for w in words if w != word]
    return sorted(sims, key=lambda t: -t[1])[:k]

for w in ["cat", "puppy", "truck", "apple"]:
    print(f"{w:>6} ->", ", ".join(f"{n} ({s:.2f})" for n, s in nearest_words(w)))

In [ ]:
# Plot the first two learned dimensions
groups = {
    "pets":     {"cat", "dog", "kitten", "puppy", "pet", "vet", "cute", "loyal", "young"},
    "vehicles": {"car", "truck", "bus", "engine", "fuel", "road", "highway", "garage", "drove", "parked"},
    "fruit":    {"apple", "banana", "orange", "fruit", "sweet", "healthy", "ate", "fresh"},
}
color_of = {w: g for g, ws in groups.items() for w in ws}
palette = {"pets": "tab:blue", "vehicles": "tab:orange", "fruit": "tab:green", None: "lightgray"}

fig, ax = plt.subplots(figsize=(8, 6))
for w in words:
    x, y = word_vectors[idx[w], 1], word_vectors[idx[w], 2]
    ax.scatter(x, y, color=palette[color_of.get(w)], s=40)
    ax.annotate(w, (x, y), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("learned dim 2"); ax.set_ylabel("learned dim 3")
ax.set_title("Word vectors learned from co-occurrence counts + SVD")
plt.show()

No one told the algorithm that *kitten* relates to *cat*, or *truck* to *car*.
It worked that out purely from **which words appear near which**. That's the
leap from hand-made features to **learned** embeddings.

(We plot dimensions 2 and 3 because dimension 1 of an SVD mostly captures
"how frequent is this word" rather than meaning.)

With a tiny corpus the result is noisy. Real models learn from billions of
words, and modern ones use **transformers** that look at a word's whole
sentence, so "bank" in *river bank* and "bank" in *bank account* get
different vectors.

> **Try it:** change `WINDOW` to `1` and to `8`, and `DIMS` to `2` and `10`.
> How do the nearest neighbours change? Then add a few sentences about a new
> topic (say, music) and see whether a new cluster forms.

---
## Part 4: Sentence embeddings with a pretrained model

Now for the real thing. [`sentence-transformers`](https://www.sbert.net/) wraps
pretrained transformer models that turn a **whole sentence** into one vector.

We'll use **`all-MiniLM-L6-v2`**:
- 384 dimensions
- about 90 MB, fast on a laptop CPU
- trained on over 1 billion sentence pairs to put paraphrases close together

The first run downloads the model. After that it loads from your local cache.

**Offline fallback:** if the download fails, the cell below switches to a
simple character n-gram hashing embedder so every cell still runs. It matches
spelling, not meaning, so the "no shared words" examples won't work in that
mode. Comparing the two is itself a good lesson in what a trained model adds.

In [ ]:
from sklearn.feature_extraction.text import HashingVectorizer

MODEL_NAME = "all-MiniLM-L6-v2"

try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(MODEL_NAME)

    def embed(texts):
        # normalize_embeddings=True -> unit vectors, so dot product == cosine
        return model.encode(list(texts), normalize_embeddings=True)

    BACKEND = f"sentence-transformers / {MODEL_NAME}"
except Exception as err:  # no internet, blocked download, missing torch, ...
    print(f"[fallback] Could not load {MODEL_NAME}: {type(err).__name__}")
    # Stateless and needs no training: hash character 3- and 4-grams into a fixed
    # 4096-D vector. It catches shared word *pieces* (stretch ~ stretching)
    # but not true meaning (sofa ~ couch), so expect weaker results.
    _hasher = HashingVectorizer(analyzer="char_wb", ngram_range=(3, 4), n_features=2**12,
                                alternate_sign=False, norm="l2")

    def embed(texts):
        return _hasher.transform(list(texts)).toarray()

    BACKEND = "fallback: hashed character n-grams"

print("Embedding backend:", BACKEND)

In [ ]:
sentences = [
    "A man is playing a guitar.",
    "Someone is strumming an acoustic guitar.",
    "A woman is slicing vegetables in the kitchen.",
    "A chef is chopping onions.",
    "The stock market fell sharply today.",
    "Shares dropped as investors panicked.",
    "A cat is sleeping on the sofa.",
    "The kitten naps on the couch.",
]
E = embed(sentences)
print("Embeddings shape:", E.shape, "-> (num_sentences, dimensions)")
print("First 8 numbers of sentence 0:", E[0][:8])
print("Vector length (should be ~1.0):", np.linalg.norm(E[0]).round(4))

In [ ]:
short = [s if len(s) < 34 else s[:31] + "..." for s in sentences]
plot_similarity(E @ E.T, short, f"Sentence similarity ({BACKEND})")

Read the heatmap. The bright 2x2 blocks along the diagonal are the
**paraphrase pairs**:

- *"playing a guitar"* ≈ *"strumming an acoustic guitar"*
- *"slicing vegetables"* ≈ *"chopping onions"*: **no shared content words**, yet they're similar
- *"stock market fell"* ≈ *"shares dropped"*: no shared words either
- *"cat sleeping on the sofa"* ≈ *"kitten naps on the couch"*

The model has learned that *sofa ≈ couch* and *fell ≈ dropped* from context,
at a scale far beyond our Part 3 toy.

> **Try it:** add a sentence that is a **negation** ("A man is *not* playing a
> guitar."). Is it close to or far from the original? *(Spoiler: embedding models
> are often surprisingly bad at negation. Worth knowing before you build on them.)*

---
## Part 5: Semantic search

Semantic search is the most common use of embeddings, and it's the "R" in
RAG. The recipe:

1. **Index:** embed every document once and store the vectors.
2. **Query:** embed the user's question with the *same* model.
3. **Rank:** score every document by cosine similarity and return the top *k*.

In [ ]:
docs = [
    # cooking
    "How to bake sourdough bread at home with a starter",
    "Quick weeknight pasta recipes with garlic and olive oil",
    "Tips for grilling the perfect steak",
    "A beginner's guide to making sushi rolls",
    # fitness
    "Couch to 5K: a running plan for complete beginners",
    "Strength training exercises you can do without a gym",
    "Why stretching after a workout reduces soreness",
    "How much protein do you need to build muscle",
    # tech
    "Getting started with Python for data analysis",
    "How to set up a home Wi-Fi mesh network",
    "Understanding how large language models generate text",
    "Keyboard shortcuts that will speed up your coding",
    # travel
    "Backpacking through Southeast Asia on a budget",
    "The best time of year to visit Japan for cherry blossoms",
    "Packing checklist for a week-long hiking trip",
    "How to beat jet lag on long-haul flights",
    # finance
    "Index funds versus picking individual stocks",
    "How to build an emergency savings fund",
    "Understanding compound interest with simple examples",
    "Paying off credit card debt: avalanche vs snowball",
]
topics = ["cooking"] * 4 + ["fitness"] * 4 + ["tech"] * 4 + ["travel"] * 4 + ["finance"] * 4

doc_vectors = embed(docs)   # step 1: index (done once)
print("Indexed", len(docs), "documents ->", doc_vectors.shape)

In [ ]:
def search(query, k=3):
    q = embed([query])[0]                      # step 2: embed the query
    scores = doc_vectors @ q                   # step 3: cosine (vectors are unit length)
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), docs[i]) for i in top]

def show(query, k=3):
    print(f"\n🔎 {query}")
    for score, doc in search(query, k):
        print(f"   {score:.3f}  {doc}")

show("I want to get fitter but I've never jogged before")
show("what should I eat after lifting weights")
show("saving money for retirement")
show("feeling tired after flying across time zones")

### Semantic vs. keyword search

Compare this with a plain **keyword** search that counts shared words. Look at
queries that share *no* words with the right answer:

In [ ]:
import re

def tokenize(s):
    return set(re.findall(r"[a-z0-9]+", s.lower()))

def keyword_search(query, k=3):
    q = tokenize(query)
    scores = [len(q & tokenize(d)) for d in docs]
    top = np.argsort(scores)[::-1][:k]
    return [(scores[i], docs[i]) for i in top]

for query in ["recovering from muscle aches", "cheap trip abroad", "AI chatbots explained"]:
    print(f"\n🔎 {query}")
    print("   keyword :", keyword_search(query, 1)[0])
    print("   semantic:", tuple(round(x, 3) if isinstance(x, float) else x for x in search(query, 1)[0]))

Keyword search can only latch onto an incidental shared word (*muscle*,
*trip*) or, with zero overlap, return an arbitrary result. With the
pretrained model, semantic search matches **meaning**: *muscle aches → soreness*, *cheap trip →
budget backpacking*, *AI chatbots → language models*.

In production you'd often **combine both** (*hybrid search*), because keywords
still win for exact names, IDs and rare jargon.

> **Try it:** add 5 documents about a hobby of yours and search them. Then find
> a query where semantic search gets it **wrong**. What does that tell you?

---
## Part 6: Visualizing embeddings

People can't picture 384 dimensions. **Dimensionality reduction** squashes
vectors down to 2-D while trying to keep the neighbourhood structure intact:

- **PCA** is linear and fast. It keeps the directions of greatest variance, so the *global* layout is trustworthy.
- **t-SNE** is non-linear. It's great for revealing *local* clusters, but distances *between* clusters don't mean much.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

topic_names = sorted(set(topics))
topic_colors = dict(zip(topic_names, plt.cm.tab10.colors))

pca_2d = PCA(n_components=2, random_state=0).fit_transform(doc_vectors)
tsne_2d = TSNE(n_components=2, perplexity=5, random_state=0, init="pca").fit_transform(doc_vectors)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, pts, title in [(axes[0], pca_2d, "PCA"), (axes[1], tsne_2d, "t-SNE")]:
    for t in topic_names:
        m = np.array(topics) == t
        ax.scatter(pts[m, 0], pts[m, 1], s=70, color=topic_colors[t], label=t)
    for (x, y), d in zip(pts, docs):
        ax.annotate(d[:28] + "...", (x, y), fontsize=7, xytext=(4, 2), textcoords="offset points")
    ax.set_title(f"{title} of {doc_vectors.shape[1]}-D document embeddings")
axes[0].legend(loc="best")
plt.tight_layout()
plt.show()

Documents from the same topic should cluster together. **The model was never
told the topics.** The structure comes from the embeddings alone.

### Clustering: letting the data find the topics

If embeddings group by meaning, an unsupervised algorithm like **k-means**
should be able to recover the topics without labels.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

kmeans = KMeans(n_clusters=5, n_init=10, random_state=0).fit(doc_vectors)
for c in range(5):
    members = [d for d, lbl in zip(docs, kmeans.labels_) if lbl == c]
    print(f"\nCluster {c}:")
    for d in members:
        print("   -", d)

ari = adjusted_rand_score(topics, kmeans.labels_)
print(f"\nAgreement with the true topics (Adjusted Rand Index, 1.0 = perfect): {ari:.2f}")

---
## Recap

| Concept | What we saw |
|---|---|
| Embedding | A vector where **geometry = meaning** |
| Cosine similarity | Compares the **angle**. It's the standard metric for text |
| One-hot | Sparse and orthogonal, so it captures **no** similarity |
| Learned vectors | Come from **context** (distributional hypothesis), via counts + SVD or neural nets |
| Sentence models | Pretrained transformers embed whole sentences and match paraphrases with no shared words |
| Semantic search | Embed → compare → rank. The foundation of RAG |
| Visualization | PCA and t-SNE project to 2-D. Clusters appear without labels |

---
## Exercises

Do these in new cells below, or in your own notebook. Tick them off in the
chapter README as you go.

1. **`most_similar`:** implement `most_similar(name, k)` for the Part 1 items. Return the `k` nearest by cosine similarity, excluding the item itself.
2. **Metric showdown:** rank the Part 5 documents for one query using cosine similarity, raw dot product *without* normalization, and Euclidean distance. When do the rankings differ, and why do they agree once vectors are normalized?
3. **Word arithmetic:** the classic demo is `king - man + woman ≈ queen`. Try `embed(["king"]) - embed(["man"]) + embed(["woman"])` with the sentence model and search a list of candidate words. Does it work? Why might a *sentence* model be worse at this than word2vec?
4. **Break it:** find three queries where semantic search returns the wrong top result, for example negation, numbers, or rare names. Write one sentence about why each fails.
5. **Your own corpus:** replace `docs` with 30+ lines from something you care about (notes, song titles, commit messages). Embed, plot with t-SNE, and cluster. Do the clusters make sense?
6. **Hybrid search:** combine `keyword_search` and `search` scores, e.g. `0.3 * keyword + 0.7 * semantic` after scaling both to [0, 1]. Find a query where hybrid beats both.
7. **Compare models:** swap `MODEL_NAME` for `"all-mpnet-base-v2"` (768-D, slower, usually better) and re-run Part 5. What changes in speed, dimensions, and quality?
8. **Stretch: brute force vs. index.** Time `search()` over 10,000 random unit vectors, then over 1,000,000. At what size does brute force feel slow? (This is the motivation for the next chapter, **Vector Databases**.)

---
## Further reading

- Jay Alammar, [*The Illustrated Word2vec*](https://jalammar.github.io/illustrated-word2vec/): the best visual intro to learned word vectors
- [Sentence-Transformers documentation](https://www.sbert.net/): models, fine-tuning, and semantic search recipes
- [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard): how embedding models compare across tasks
- Mikolov et al., 2013, *Efficient Estimation of Word Representations in Vector Space* (the word2vec paper)